<a href="https://colab.research.google.com/github/anjaliii210/LLM-Post-Training-with-QLoRA-SFT-DPO/blob/main/1_LoRA_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U bitsandbytes accelerate

In [ ]:
import torch
import bitsandbytes as bnb
from transformers.utils import is_bitsandbytes_available

print(bnb.__version__)
print(is_bitsandbytes_available())
print(torch.cuda.is_available())

0.50.2
True
True


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

print("Model and tokenizer loaded successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model and tokenizer loaded successfully!


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [ ]:
in_features = 896
out_features = 896

linear = nn.Linear(in_features, out_features, bias=False)

print("Original parameters:", sum(
    p.numel() for p in linear.parameters()
))

Original parameters: 802816


In [ ]:
#implementing LoRA
class LoRALinear(nn.Module):

    def __init__(self, in_features, out_features,
                 rank=16, alpha=32):

        super().__init__()

        self.linear = nn.Linear(
            in_features, out_features, bias=False
        )

        self.rank = rank
        self.scaling = alpha / rank

        # Freeze original weights
        self.linear.weight.requires_grad = False

        # Trainable low-rank matrices
        self.A = nn.Parameter(
            torch.randn(rank, in_features) * 0.01
        )

        self.B = nn.Parameter(
            torch.zeros(out_features, rank)
        )

    def forward(self, x):

        original_output = self.linear(x)

        lora_update = (
            x @ self.A.T @ self.B.T
        ) * self.scaling

        return original_output + lora_update

In [ ]:
#testing implementation
lora_layer = LoRALinear(
    in_features=896,
    out_features=896,
    rank=16,
    alpha=32
)

x = torch.randn(2, 10, 896)

output = lora_layer(x)

print("Input:", x.shape)
print("Output:", output.shape)

for name, param in lora_layer.named_parameters():
    print(
        name,
        param.shape,
        "Trainable:", param.requires_grad
    )

Input: torch.Size([2, 10, 896])
Output: torch.Size([2, 10, 896])
A torch.Size([16, 896]) Trainable: True
B torch.Size([896, 16]) Trainable: True
linear.weight torch.Size([896, 896]) Trainable: False


In [ ]:
total_params = sum(
    p.numel() for p in lora_layer.parameters()
)

trainable_params = sum(
    p.numel() for p in lora_layer.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

print(
    "Trainable percentage:",
    round(trainable_params / total_params * 100, 2),
    "%"
)

Total parameters: 831488
Trainable parameters: 28672
Trainable percentage: 3.45 %


In [ ]:
optimizer = torch.optim.AdamW(
    [lora_layer.A, lora_layer.B],
    lr=1e-3
)

target = torch.randn(2, 10, 896)

optimizer.zero_grad()

prediction = lora_layer(x)
loss = F.mse_loss(prediction, target)

loss.backward()

print("Loss:", loss.item())
print("A gradient:", lora_layer.A.grad.norm().item())
print("B gradient:", lora_layer.B.grad.norm().item())
print(
    "Original weight gradient:",
    lora_layer.linear.weight.grad
)

optimizer.step()

Loss: 1.323357343673706
A gradient: 0.0
B gradient: 0.04335326328873634
Original weight gradient: None


LoRA to Qwen Model

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

Configuring LoRA

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

In [ ]:
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
for name, param in model.named_parameters():

    if param.requires_grad:
        print(name, param.shape)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([16, 896])
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([896, 16])
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.Size([16, 896])
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.Size([128, 16])
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([16, 896])
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.Size([128, 16])
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight torch.Size([16, 896])
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight torch.Size([896, 16])
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight torch.Size([16, 896])
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight torch.Size([896, 16])
base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight

In [ ]:
expected = 24 * (
    2 * 16 * (896 + 896) +
    2 * 16 * (896 + 128)
)

actual = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Expected LoRA parameters:", expected)
print("Actual trainable parameters:", actual)

Expected LoRA parameters: 2162688
Actual trainable parameters: 2162688


In [ ]:
#testing adapted model
model.config.use_cache = True
model.eval()

messages = [
    {
        "role": "user",
        "content": "Explain overfitting in machine learning."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

print(tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
))

#adapters newly initialised
#not fine-tuned

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain overfitting in machine learning.
assistant
Overfitting is a common issue that occurs when a model is trained on too much data while ignoring the noise and variability in the training set. This can lead to poor performance on unseen data, as well as increased variance in predictions.

Here's how overfitting happens:

1. **High Dimensionality**: The dataset has many more features than there are samples (or observations). This makes it difficult for the model to generalize well.
2. **Random Initialization**: If the initial weights of the model are randomly initialized, they may not be good choices due to their sensitivity to initialization.
3. **


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/LLM_PostTraining/qlora_adapter"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("LoRA adapter and tokenizer saved!")

LoRA adapter and tokenizer saved!


In [ ]:
import os

print(os.listdir(SAVE_PATH))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
